# Stage 2: Cross-Encoder Re-ranking
## BGE Retrieval → Re-rank top-50 → Final Results

**What this notebook does:**

This is the Retrieve-then-Rerank pipeline — the current state-of-the-art approach in IR.

```
Stage 1 (already done): BGE embeds query → FAISS finds top-1000 companies (fast, semantic)
                                    ↓
Stage 2 (this notebook): Cross-encoder reads query + each company → scores relevance (precise)
                                    ↓
                         Final re-ranked top-50 results
```

**Why two stages?**
- BGE (bi-encoder) is fast but encodes query and company SEPARATELY — less precise
- Cross-encoder reads query + company TOGETHER like a human — much more precise
- But cross-encoder is slower so we only run it on BGE's top-50 candidates

**Model used:** `BAAI/bge-reranker-v2-m3`
- From the same research group as BGE-large — designed to work together
- Multilingual, strong on English
- ~570M parameters

**Folder structure:**
```
result/
└── reranker/
    ├── reranked_results.csv        # Final re-ranked results for all 101 queries
    ├── evaluation_reranker.csv     # Full metrics: Precision, Recall, NDCG @k
    ├── latency_reranker.csv        # Per-query latency breakdown
    ├── comparison_final.csv        # Side-by-side vs all baselines
    └── reranker_plots.png          # Quality + latency visualisations
```

## 1 · Environment Setup

In [ ]:
import os, json, time
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from dotenv import load_dotenv
from transformers import AutoModelForSequenceClassification, AutoTokenizer

load_dotenv()

# ── GPU check ─────────────────────────────────────────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'[Setup] Device        : {DEVICE}')
if torch.cuda.is_available():
    print(f'[Setup] GPU           : {torch.cuda.get_device_name(0)}')
    print(f'[Setup] VRAM          : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('[Setup] WARNING: No GPU detected — re-ranking will be slow on CPU')

# ── Output folder ─────────────────────────────────────────────────────────────
RESULT_DIR = Path('result/05_reranker')
RESULT_DIR.mkdir(parents=True, exist_ok=True)
print(f'[Setup] Result folder : {RESULT_DIR}/')

# ── Key settings ──────────────────────────────────────────────────────────────
# How many BGE candidates to pass to the cross-encoder
# 50 is a good balance: enough for good recall, fast enough for re-ranking
TOP_K_RERANK = 50
print(f'[Setup] Re-ranking top-{TOP_K_RERANK} BGE candidates per query')

## 2 · Load BGE Retrieval Results & Corpus

We reuse the BGE retrieval results already computed in the baseline notebook.
No need to re-run BGE — we just re-rank its top-50 candidates.

In [ ]:
print('[Load] Loading data...')

# ── BGE retrieval results (from baseline experiment) ──────────────────────────
bge_df = pd.read_csv('result/3_baseline_BGE/embedding_results.csv')
print(f'[Load] BGE results    : {len(bge_df):,} rows  ({bge_df["query_id"].nunique()} queries)')

# ── Company corpus with full fields ───────────────────────────────────────────
all_companies = pd.read_csv('result/company_corpus.csv')
print(f'[Load] Corpus size    : {len(all_companies):,} companies')
print(f'[Load] Corpus columns : {list(all_companies.columns)}')

# ── Production results for evaluation ─────────────────────────────────────────
production_df = pd.read_excel('dataset/production_results.xlsx')
print(f'[Load] Production rows: {len(production_df):,}')

# ── Queries ───────────────────────────────────────────────────────────────────
with open('dataset/goi_search_results.json', 'r') as f:
    data = json.load(f)
print(f'[Load] Queries        : {len(data)}')

# ── Sanity check: show sample BGE result ─────────────────────────────────────
sample_query = bge_df['query'].iloc[0]
sample_top5  = bge_df[bge_df['query'] == sample_query].head(5)[['rank','name','score']]
print(f'\n[Load] Sample query: "{sample_query}"')
print(f'[Load] BGE top-5:')
print(sample_top5.to_string(index=False))

## 3 · Load Cross-Encoder Model

**Model: `BAAI/bge-reranker-v2-m3`**

Unlike BGE (bi-encoder) which encodes query and company separately,
the cross-encoder reads BOTH together:

```
Input:  [query] [SEP] [company summary]
Output: single relevance score (higher = more relevant)
```

This is more accurate because the model can see exactly how the query
and company relate to each other — not just their individual meanings.

In [ ]:
print('[Model] Loading cross-encoder model...')
print('[Model] Model: BAAI/bge-reranker-v2-m3')
print('[Model] This may take 1-2 minutes on first run (downloading ~570MB)...')

t0 = time.time()

RERANKER_MODEL = 'BAAI/bge-reranker-v2-m3'

tokenizer = AutoTokenizer.from_pretrained(RERANKER_MODEL)
reranker  = AutoModelForSequenceClassification.from_pretrained(RERANKER_MODEL)
reranker  = reranker.to(DEVICE)
reranker.eval()  # disable dropout for inference

load_time = time.time() - t0
print(f'[Model] Loaded in {load_time:.1f}s')
print(f'[Model] Model device: {next(reranker.parameters()).device}')
print(f'[Model] Parameters  : {sum(p.numel() for p in reranker.parameters())/1e6:.0f}M')

## 4 · Re-ranking Function

The cross-encoder scores each (query, company_summary) pair.
We batch the pairs for efficiency — processing multiple at once on GPU.

**How it works:**
1. Take top-K candidates from BGE (default: top-50)
2. Create pairs: [(query, summary_1), (query, summary_2), ...]
3. Run all pairs through cross-encoder in one batched call
4. Sort by cross-encoder score (not BGE score)
5. Return re-ranked list

In [ ]:
def rerank_candidates(query, candidates_df, top_k=TOP_K_RERANK, batch_size=32):
    """
    Re-rank candidate companies using cross-encoder.
    
    Args:
        query         : the search query string
        candidates_df : dataframe of BGE top-k results for this query
        top_k         : how many candidates to re-rank
        batch_size    : how many pairs to score at once on GPU
    
    Returns:
        dataframe sorted by cross-encoder score (highest first)
    """
    # Take only top-K candidates
    candidates = candidates_df.head(top_k).copy()
    summaries  = candidates['summary'].fillna('').tolist()

    # Build (query, summary) pairs
    pairs = [[query, summary] for summary in summaries]

    all_scores = []

    # Score in batches
    with torch.no_grad():  # no gradient computation needed for inference
        for i in range(0, len(pairs), batch_size):
            batch = pairs[i:i + batch_size]

            # Tokenize: query and summary concatenated with [SEP]
            encoded = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors='pt'
            ).to(DEVICE)

            # Forward pass — output is logit (raw score, not probability)
            scores = reranker(**encoded).logits.squeeze(-1)
            all_scores.extend(scores.cpu().float().tolist())

    # Add cross-encoder scores to dataframe
    candidates['reranker_score'] = all_scores
    candidates['bge_rank']       = candidates['rank']  # keep original BGE rank

    # Sort by reranker score (descending) and assign new ranks
    reranked = candidates.sort_values('reranker_score', ascending=False).reset_index(drop=True)
    reranked['rank'] = reranked.index + 1

    return reranked

# ── Smoke test on one query ───────────────────────────────────────────────────
print('[Test] Running smoke test on first query...')
test_item       = data[0]
test_query      = test_item['query']
test_candidates = bge_df[bge_df['query_id'] == test_item['query_id']].head(TOP_K_RERANK)

t0          = time.time()
test_result = rerank_candidates(test_query, test_candidates)
test_time   = (time.time() - t0) * 1000

print(f'[Test] Query          : "{test_query}"')
print(f'[Test] Candidates in  : {len(test_candidates)}')
print(f'[Test] Time taken     : {test_time:.0f}ms')
print(f'[Test] Re-ranked top-5 (BGE rank → Reranker rank):')
comparison = test_result[['rank','bge_rank','name','reranker_score']].head(10)
print(comparison.to_string(index=False))

print('\n[Test] Smoke test passed! ✅')

## 5 · Re-rank All 101 Queries

Run the full re-ranking pipeline across all queries.
We log progress every 10 queries so you can monitor speed.

In [ ]:
print(f'[Rerank] Starting re-ranking for all {len(data)} queries...')
print(f'[Rerank] Re-ranking top-{TOP_K_RERANK} BGE candidates per query')
print(f'[Rerank] Model: BAAI/bge-reranker-v2-m3 on {DEVICE}')
print('-' * 60)

all_reranked_rows = []
latency_rows      = []
total_start       = time.time()

for i, item in enumerate(data):
    qid   = item['query_id']
    query = item['query']

    # ── Get BGE top-K candidates for this query ───────────────────────────────
    candidates = (
        bge_df[bge_df['query_id'] == qid]
        .sort_values('rank')
        .head(TOP_K_RERANK)
    )

    if len(candidates) == 0:
        print(f'[Rerank] WARNING: No BGE results found for query_id={qid} ("{query}")')
        continue

    # ── Re-rank ───────────────────────────────────────────────────────────────
    t0       = time.perf_counter()
    reranked = rerank_candidates(query, candidates, top_k=TOP_K_RERANK)
    rerank_ms = (time.perf_counter() - t0) * 1000

    # ── Store results ─────────────────────────────────────────────────────────
    for _, row in reranked.iterrows():
        all_reranked_rows.append({
            'query_id':       qid,
            'query':          query,
            'rank':           int(row['rank']),
            'bge_rank':       int(row['bge_rank']),
            'reranker_score': float(row['reranker_score']),
            'bge_score':      float(row['score']),
            'domain':         row['domain'],
            'name':           row['name'],
            'summary':        row['summary'],
        })

    latency_rows.append({
        'query_id':    qid,
        'query':       query,
        'n_candidates': len(candidates),
        'rerank_ms':   round(rerank_ms, 1),
    })

    # ── Progress log every 10 queries ─────────────────────────────────────────
    if (i + 1) % 10 == 0 or (i + 1) == len(data):
        elapsed    = time.time() - total_start
        avg_ms     = elapsed / (i + 1) * 1000
        remaining  = (len(data) - i - 1) * elapsed / (i + 1)
        print(f'[Rerank] {i+1:3d}/{len(data)} queries done  |  '
              f'avg {avg_ms:.0f}ms/query  |  '
              f'~{remaining:.0f}s remaining')

total_time = time.time() - total_start
print('-' * 60)
print(f'[Rerank] All queries done!')
print(f'[Rerank] Total time    : {total_time:.1f}s')
print(f'[Rerank] Avg per query : {total_time/len(data)*1000:.0f}ms')

# ── Save results ──────────────────────────────────────────────────────────────
reranked_df = pd.DataFrame(all_reranked_rows)
latency_df  = pd.DataFrame(latency_rows)

reranked_df.to_csv(RESULT_DIR / 'reranked_results.csv', index=False)
latency_df.to_csv(RESULT_DIR  / 'latency_reranker.csv', index=False)

print(f'[Rerank] Results saved : result/reranker/reranked_results.csv')
print(f'[Rerank] Latency saved : result/reranker/latency_reranker.csv')

## 6 · Qualitative Check — Did Re-ranking Fix the Investment Bank Problem?

Before running full evaluation, do a sanity check on the "software companies" query.
This is the query where BGE was returning investment banks.

Look at:
- Which companies moved UP after re-ranking (cross-encoder thought were more relevant)
- Which companies moved DOWN (cross-encoder correctly penalised them)

In [ ]:
# ── Find the software companies query ────────────────────────────────────────
target_queries = ['software companies', 'software', 'software firms']
soft_query = None
for item in data:
    if item['query'].lower().strip() in target_queries:
        soft_query = item
        break

if soft_query is None:
    # Just use the first query as example
    soft_query = data[0]
    print(f'[Check] "software companies" not found — using: "{soft_query["query"]}"')
else:
    print(f'[Check] Found target query: "{soft_query["query"]}"')

qid = soft_query['query_id']

# ── BGE top-10 (before re-ranking) ───────────────────────────────────────────
bge_top10 = bge_df[bge_df['query_id'] == qid].sort_values('rank').head(10)
print(f'\n[Check] BGE top-10 (BEFORE re-ranking):')
print(f'{"Rank":<6} {"Name":<45} {"BGE Score"}')
print('-' * 65)
for _, row in bge_top10.iterrows():
    print(f'{int(row["rank"]):<6} {str(row["name"])[:43]:<45} {row["score"]:.4f}')

# ── Re-ranked top-10 (after re-ranking) ──────────────────────────────────────
reranked_top10 = reranked_df[reranked_df['query_id'] == qid].sort_values('rank').head(10)
print(f'\n[Check] Re-ranked top-10 (AFTER re-ranking):')
print(f'{"New Rank":<10} {"Old BGE Rank":<14} {"Name":<45} {"Reranker Score"}')
print('-' * 80)
for _, row in reranked_top10.iterrows():
    movement = int(row['bge_rank']) - int(row['rank'])
    arrow    = f'(↑{movement})' if movement > 0 else f'(↓{abs(movement)})' if movement < 0 else '(=)'
    print(f'{int(row["rank"]):<10} {int(row["bge_rank"]):<14} {str(row["name"])[:43]:<45} {row["reranker_score"]:.4f} {arrow}')

# ── Companies that moved the most ─────────────────────────────────────────────
merged = reranked_df[reranked_df['query_id'] == qid].copy()
merged['rank_change'] = merged['bge_rank'] - merged['rank']

print(f'\n[Check] Biggest rank improvements (moved UP):')
top_movers = merged.nlargest(5, 'rank_change')[['name','bge_rank','rank','rank_change','reranker_score']]
print(top_movers.to_string(index=False))

print(f'\n[Check] Biggest rank drops (moved DOWN — likely investment banks!):')
worst_movers = merged.nsmallest(5, 'rank_change')[['name','bge_rank','rank','rank_change','reranker_score']]
print(worst_movers.to_string(index=False))

## 7 · Evaluation — Precision, Recall, NDCG@k

Evaluate the re-ranked results using the same metrics and pseudo-relevance labels
as all baseline experiments.

**Important note on k values:**
Since we only re-rank the top-50 BGE candidates, we can only meaningfully
evaluate at k ≤ 50. At k=100 or k=1000 we do not have re-ranked results —
for those positions we fall back to the original BGE ranking.
This is noted clearly in the results.

In [ ]:
print('[Eval] Starting evaluation...')

K_VALUES = [10, 50, 100, 1000]

def get_relevant(query_id, top_k=100):
    return set(production_df[
        (production_df['query_id'] == query_id) &
        (production_df['rank'] <= top_k)
    ]['domain'].tolist())

def precision_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & relevant) / k if k else 0

def recall_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & relevant) / len(relevant) if relevant else 0

def dcg_at_k(retrieved, relevant, k):
    return sum(1 / np.log2(i + 2) for i, d in enumerate(retrieved[:k]) if d in relevant)

def ndcg_at_k(retrieved, relevant, k):
    ideal = dcg_at_k(list(relevant), relevant, k)
    return dcg_at_k(retrieved, relevant, k) / ideal if ideal else 0

eval_rows = []
n_queries = len(data)

for i, item in enumerate(data):
    qid      = item['query_id']
    query    = item['query']
    relevant = get_relevant(qid)

    # ── Re-ranked top-50 domains ──────────────────────────────────────────────
    reranked_domains = (
        reranked_df[reranked_df['query_id'] == qid]
        .sort_values('rank')['domain']
        .tolist()
    )  # top-50 only

    # ── BGE full top-1000 domains (for k=100, k=1000 positions) ──────────────
    bge_all_domains = (
        bge_df[bge_df['query_id'] == qid]
        .sort_values('rank')['domain']
        .tolist()
    )  # top-1000

    # ── Build combined list: re-ranked top-50 + BGE positions 51-1000 ────────
    # This is the fairest evaluation: re-ranker improves top-50,
    # positions 51+ remain as BGE ranked them
    reranked_set   = set(reranked_domains)
    bge_remaining  = [d for d in bge_all_domains if d not in reranked_set]
    combined       = reranked_domains + bge_remaining

    for k in K_VALUES:
        eval_rows.append({
            'query_id'  : qid,
            'query'     : query,
            'k'         : k,
            'precision' : precision_at_k(combined, relevant, k),
            'recall'    : recall_at_k(combined, relevant, k),
            'ndcg'      : ndcg_at_k(combined, relevant, k),
        })

    if (i + 1) % 20 == 0:
        print(f'[Eval] {i+1}/{n_queries} queries evaluated...')

eval_df = pd.DataFrame(eval_rows)
eval_df.to_csv(RESULT_DIR / 'evaluation_reranker.csv', index=False)

print(f'[Eval] Done! Saved to result/reranker/evaluation_reranker.csv')
print(f'[Eval] Total rows: {len(eval_df):,}')

## 8 · Results Comparison — Reranker vs All Baselines

The key comparison table for your thesis.
Did the re-ranker improve over BGE alone?

In [ ]:
print('[Compare] Loading baseline results for comparison...')

# ── Load all baseline evaluation results ─────────────────────────────────────
baseline_df = pd.read_csv('result/evaluation_fixed.csv')
print(f'[Compare] Baseline methods: {baseline_df["method"].unique().tolist()}')

# ── Build comparison table ────────────────────────────────────────────────────
METHOD_ORDER = ['BM25', 'MiniLM', 'BGE', 'OpenAI', 'BGE + Reranker']

all_results = {}
for method in ['BM25', 'MiniLM', 'BGE', 'OpenAI']:
    all_results[method] = baseline_df[baseline_df['method'] == method]
all_results['BGE + Reranker'] = eval_df

print('\n' + '=' * 70)
print(f'{"Method":<22} {"k":>6} | {"NDCG":>7} | {"Prec":>7} | {"Recall":>7}')
print('=' * 70)

comparison_rows = []
for method in METHOD_ORDER:
    df = all_results[method]
    for k in K_VALUES:
        subset = df[df['k'] == k]
        ndcg   = subset['ndcg'].mean()
        prec   = subset['precision'].mean()
        rec    = subset['recall'].mean()
        print(f'{method:<22} {k:>6} | {ndcg:>7.3f} | {prec:>7.3f} | {rec:>7.3f}')
        comparison_rows.append({
            'method': method, 'k': k,
            'ndcg': round(ndcg, 3),
            'precision': round(prec, 3),
            'recall': round(rec, 3),
        })
    print('-' * 70)

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(RESULT_DIR / 'comparison_final.csv', index=False)
print(f'\n[Compare] Saved to result/reranker/comparison_final.csv')

# ── Print improvement over BGE ────────────────────────────────────────────────
print('\n[Compare] Improvement of BGE + Reranker over BGE alone:')
for k in K_VALUES:
    bge_ndcg = all_results['BGE'][all_results['BGE']['k'] == k]['ndcg'].mean()
    rer_ndcg = all_results['BGE + Reranker'][all_results['BGE + Reranker']['k'] == k]['ndcg'].mean()
    diff     = rer_ndcg - bge_ndcg
    pct      = 100 * diff / bge_ndcg if bge_ndcg > 0 else 0
    arrow    = '↑' if diff > 0 else '↓'
    print(f'  NDCG@{k:<5}: BGE={bge_ndcg:.3f}  Reranker={rer_ndcg:.3f}  {arrow}{abs(diff):.3f} ({pct:+.1f}%)')

## 9 · Latency Analysis

Break down the total query latency for the full pipeline:

```
BGE encode query  →  16.6ms   (from baseline)
BGE FAISS search  →  27.8ms   (from baseline)
Cross-encoder     →  ?ms      (measured here, depends on GPU)
─────────────────────────────
Total pipeline    →  ?ms
```

Also compare against the 44.4ms BGE baseline.

In [ ]:
print('[Latency] Analysing query latency...')

# ── Re-ranker latency stats ───────────────────────────────────────────────────
avg_rerank_ms = latency_df['rerank_ms'].mean()
min_rerank_ms = latency_df['rerank_ms'].min()
max_rerank_ms = latency_df['rerank_ms'].max()
std_rerank_ms = latency_df['rerank_ms'].std()

BGE_ENCODE_MS = 16.6   # from baseline experiment
BGE_SEARCH_MS = 27.8   # from baseline experiment
BGE_TOTAL_MS  = 44.4   # from baseline experiment

total_pipeline_ms = BGE_ENCODE_MS + BGE_SEARCH_MS + avg_rerank_ms

print('\n[Latency] === LATENCY BREAKDOWN ===')
print(f'[Latency] BGE query encode    : {BGE_ENCODE_MS:.1f}ms')
print(f'[Latency] BGE FAISS search    : {BGE_SEARCH_MS:.1f}ms')
print(f'[Latency] Cross-encoder       : {avg_rerank_ms:.1f}ms  (±{std_rerank_ms:.1f}ms, min={min_rerank_ms:.1f}, max={max_rerank_ms:.1f})')
print(f'[Latency] ─────────────────────────────────')
print(f'[Latency] BGE alone           : {BGE_TOTAL_MS:.1f}ms')
print(f'[Latency] BGE + Reranker      : {total_pipeline_ms:.1f}ms')
print(f'[Latency] Latency overhead    : +{avg_rerank_ms:.1f}ms ({100*avg_rerank_ms/BGE_TOTAL_MS:.0f}% increase)')

# ── Per-query latency distribution ───────────────────────────────────────────
print(f'\n[Latency] Slowest 5 queries:')
slowest = latency_df.nlargest(5, 'rerank_ms')[['query','n_candidates','rerank_ms']]
print(slowest.to_string(index=False))

print(f'\n[Latency] Fastest 5 queries:')
fastest = latency_df.nsmallest(5, 'rerank_ms')[['query','n_candidates','rerank_ms']]
print(fastest.to_string(index=False))

latency_summary = pd.DataFrame([{
    'Method'            : 'BGE alone',
    'Encode (ms)'       : BGE_ENCODE_MS,
    'Search (ms)'       : BGE_SEARCH_MS,
    'Reranker (ms)'     : 0,
    'Total (ms)'        : BGE_TOTAL_MS,
}, {
    'Method'            : 'BGE + Reranker',
    'Encode (ms)'       : BGE_ENCODE_MS,
    'Search (ms)'       : BGE_SEARCH_MS,
    'Reranker (ms)'     : round(avg_rerank_ms, 1),
    'Total (ms)'        : round(total_pipeline_ms, 1),
}])
latency_summary.to_csv(RESULT_DIR / 'latency_summary.csv', index=False)
print(f'\n[Latency] Summary saved to result/reranker/latency_summary.csv')

## 10 · Visualisations

Four panels showing the full picture:
1. NDCG@k — all methods including reranker
2. Precision@k — key metric Ralph asked about  
3. Recall@k — coverage
4. Latency breakdown — BGE vs BGE+Reranker

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

print('[Plot] Generating visualisations...')

colors = {
    'BM25':           '#2196F3',
    'MiniLM':         '#FF9800',
    'BGE':            '#4CAF50',
    'OpenAI':         '#9C27B0',
    'BGE + Reranker': '#F44336',
}
markers = {
    'BM25':           'o',
    'MiniLM':         's',
    'BGE':            '^',
    'OpenAI':         'D',
    'BGE + Reranker': '*',
}
# Make reranker line thicker and star marker bigger to stand out
linewidths  = {m: 2 for m in METHOD_ORDER}
markersizes = {m: 6 for m in METHOD_ORDER}
linewidths['BGE + Reranker']  = 3
markersizes['BGE + Reranker'] = 10

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for metric, ax, title in [
    ('ndcg',      axes[0], 'NDCG@k'),
    ('precision', axes[1], 'Precision@k'),
    ('recall',    axes[2], 'Recall@k'),
]:
    for method in METHOD_ORDER:
        df   = all_results[method]
        vals = [df[df['k'] == k][metric].mean() for k in K_VALUES]
        ax.plot(
            K_VALUES, vals,
            marker=markers[method],
            color=colors[method],
            label=method,
            linewidth=linewidths[method],
            markersize=markersizes[method],
            zorder=5 if method == 'BGE + Reranker' else 3,
        )
    ax.set_xscale('log')
    ax.set_xticks(K_VALUES)
    ax.set_xticklabels(K_VALUES)
    ax.set_xlabel('k (cutoff)')
    ax.set_ylabel(title)
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

# ── Latency breakdown bar chart ───────────────────────────────────────────────
ax  = axes[3]
x   = np.arange(2)
w   = 0.25
enc = [BGE_ENCODE_MS, BGE_ENCODE_MS]
sch = [BGE_SEARCH_MS, BGE_SEARCH_MS]
rer = [0, avg_rerank_ms]

b1 = ax.bar(x - w, enc, w, label='BGE Encode',   color='#42A5F5')
b2 = ax.bar(x,     sch, w, label='FAISS Search',  color='#66BB6A')
b3 = ax.bar(x + w, rer, w, label='Cross-encoder', color='#F44336')

ax.set_xticks(x)
ax.set_xticklabels(['BGE alone', 'BGE +\nReranker'])
ax.set_ylabel('Latency (ms)')
ax.set_title('Query Latency Breakdown', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)

# Add total labels on top of bars
totals  = [BGE_TOTAL_MS, total_pipeline_ms]
x_pos   = [x[0] + w, x[1] + w]
for xp, total in zip(x_pos, totals):
    ax.text(xp, total + 2, f'{total:.0f}ms',
            ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle('BGE + Cross-Encoder Reranker vs All Baselines',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(RESULT_DIR / 'reranker_plots.png', bbox_inches='tight', dpi=150)
plt.show()
print('[Plot] Saved to result/reranker/reranker_plots.png')

## 11 · Final Summary & Thesis Narrative

Auto-generated summary of key findings to help write the thesis chapter.

In [ ]:
print('[Summary] ============================================================')
print('[Summary] FINAL RESULTS SUMMARY')
print('[Summary] ============================================================')

bge_n10  = all_results['BGE'][all_results['BGE']['k'] == 10]['ndcg'].mean()
rer_n10  = all_results['BGE + Reranker'][all_results['BGE + Reranker']['k'] == 10]['ndcg'].mean()
bge_p10  = all_results['BGE'][all_results['BGE']['k'] == 10]['precision'].mean()
rer_p10  = all_results['BGE + Reranker'][all_results['BGE + Reranker']['k'] == 10]['precision'].mean()
bm25_n10 = all_results['BM25'][all_results['BM25']['k'] == 10]['ndcg'].mean()

ndcg_improvement    = 100 * (rer_n10 - bge_n10) / bge_n10
ndcg_vs_bm25        = 100 * (rer_n10 - bm25_n10) / bm25_n10
prec_improvement    = 100 * (rer_p10 - bge_p10) / bge_p10

print(f'\n[Summary] NDCG@10:')
print(f'  BM25 baseline    : {bm25_n10:.3f}')
print(f'  BGE alone        : {bge_n10:.3f}')
print(f'  BGE + Reranker   : {rer_n10:.3f}')
print(f'  vs BGE           : {ndcg_improvement:+.1f}%')
print(f'  vs BM25          : {ndcg_vs_bm25:+.1f}%')

print(f'\n[Summary] Precision@10:')
print(f'  BGE alone        : {bge_p10:.3f}')
print(f'  BGE + Reranker   : {rer_p10:.3f}')
print(f'  Improvement      : {prec_improvement:+.1f}%')

print(f'\n[Summary] Latency:')
print(f'  BGE alone        : {BGE_TOTAL_MS:.1f}ms')
print(f'  BGE + Reranker   : {total_pipeline_ms:.1f}ms')
print(f'  Overhead         : +{avg_rerank_ms:.1f}ms')

print(f'\n[Summary] Thesis narrative:')
print(f'  Adding a cross-encoder re-ranker on top of BGE retrieval')
print(f'  improved NDCG@10 from {bge_n10:.3f} to {rer_n10:.3f} ({ndcg_improvement:+.1f}%),')
print(f'  and Precision@10 from {bge_p10:.3f} to {rer_p10:.3f} ({prec_improvement:+.1f}%).')
print(f'  Total query latency increased from {BGE_TOTAL_MS:.1f}ms to {total_pipeline_ms:.1f}ms.')
print(f'  Overall vs BM25 baseline: NDCG@10 improved by {ndcg_vs_bm25:.1f}%.')
print('[Summary] ============================================================')